In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

def count_images_in_classes(main_folder):
    # Dictionary to hold the count of images in each class
    class_image_counts = {}

    # List all subdirectories in the main folder (each subdirectory represents a class)
    class_folders = [d for d in os.listdir(main_folder) if os.path.isdir(os.path.join(main_folder, d))]

    # Iterate through each class folder
    for class_folder in class_folders:
        class_path = os.path.join(main_folder, class_folder)

        # List all files in the class folder
        files = os.listdir(class_path)

        # Filter out image files based on common image extensions
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff'))]

        # Count the number of image files
        class_image_counts[class_folder] = len(image_files)

    return class_image_counts

# Define the path to your main folder
main_folder_path = "/content/drive/MyDrive/Systems Files/Final_Balanced_augmented_dataset"

# Get the image counts
image_counts = count_images_in_classes(main_folder_path)

# Print the results
for class_name, count in image_counts.items():
    print(f"Class '{class_name}' has {count} images.")


Class 'ModerateDemented' has 3200 images.
Class 'NonDemented' has 3200 images.
Class 'VeryMildDemented' has 3200 images.
Class 'MildDemented' has 3200 images.


In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
import numpy as np
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.models import Model


In [ ]:
target_size = (128, 128)
batch_size = 16
CHANNELS = 3

In [ ]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
            r"/content/drive/MyDrive/Systems Files/Final_Balanced_augmented_dataset",
            shuffle = True,
            image_size = (target_size),
            batch_size = (batch_size)

    )

Found 12800 files belonging to 4 classes.


In [ ]:
class_names = dataset.class_names
class_names

['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

In [ ]:
train_ds, val_ds, test_ds = dataset_splitting(dataset)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def split_directory_data(directory, train_split=0.7, test_split=0.2, val_split=0.1, target_size=(150, 150), batch_size=32):
    # Initialize an ImageDataGenerator with rescaling
    datagen = ImageDataGenerator(rescale=1./255)

    train_generator = datagen.flow_from_directory(
        directory,
        target_size=target_size,
        batch_size=batch_size,
        class_mode='SparseCategoricalCrossentropy',  # Change this according to your problem
        shuffle=True,
        seed=42
    )

    total_samples = len(train_generator.filenames)
    train_samples = int(total_samples * train_split)
    val_samples = int(total_samples * val_split)

    train_data_generator = datagen.flow_from_directory(
        directory,
        target_size=target_size,
        batch_size=batch_size,
        class_mode='SparseCategoricalCrossentropy',  # Change this according to your problem
        subset='training',
        shuffle=True,
        seed=42
    )

    val_data_generator = datagen.flow_from_directory(
        directory,
        target_size=target_size,
        batch_size=batch_size,
        class_mode='SparseCategoricalCrossentropy',  # Change this according to your problem
        subset='validation',
        shuffle=True,
        seed=42
    )

    test_data_generator = datagen.flow_from_directory(
        directory,
        target_size=target_size,
        batch_size=batch_size,
        class_mode='SparseCategoricalCrossentropy',  # Change this according to your problem
        subset='validation',
        shuffle=True,
        seed=42
    )

    return train_data_generator, val_data_generator, test_data_generator


In [ ]:
train_data_generator, val_data_generator, test_data_generator = dataset_splitting(dataset)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.models import Model
import tensorflow as tf

# Set your learning rate here
lr = 0.001
# Set the path to save the model
save_loc = 'preview/model'

# Initialize the ResNet50 model
resnet = ResNet50(include_top=False, input_shape=(128, 128, 3), pooling='max', weights='imagenet')

# Add custom layers
x = resnet.layers[-1].output
x = Dropout(0.5)(x)
x = BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001)(x)
predictions = Dense(5, activation='softmax')(x)
model = Model(inputs=resnet.input, outputs=predictions)

# Set all layers to be trainable
for layer in model.layers:
    layer.trainable = True

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adamax(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])

94765736/94765736 [==============================] - 1s 0us/step


In [ ]:
model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 128, 128, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 134, 134, 3)          0         ['input_1[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 64, 64, 64)           9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 64, 64, 64)           256       ['conv1_conv[0][0]']          
 on)                                                                                          

In [ ]:
learning_rate = 0.0001
optimizer = Adam(learning_rate=learning_rate)

model.compile(
    optimizer=optimizer,
   loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data_generator,
    batch_size=batch_size,
    validation_data=val_data_generator,
    verbose=1,
    epochs=5,
)

Epoch 1/5
640/640 [==============================] - 2065s 134ms/step - loss: 0.5311 - accuracy: 0.7902 - val_loss: 0.1829 - val_accuracy: 0.9141
Epoch 2/5
640/640 [==============================] - 119s 131ms/step - loss: 0.2788 - accuracy: 0.8866 - val_loss: 0.1221 - val_accuracy: 0.9539
Epoch 3/5
640/640 [==============================] - ETA: 0s - loss: 0.1838 - accuracy: 0.9273

In [ ]:
import matplotlib.pyplot as plt

# Assuming the 'history' variable is the output from model.fit
history_dict = history.history

# Extracting data
epochs = range(1, len(history_dict['loss']) + 1)
train_loss = history_dict['loss']
val_loss = history_dict['val_loss']
train_acc = history_dict.get('accuracy')  # Depending on the metric used
val_acc = history_dict.get('val_accuracy')  # Depending on the metric used

# Plotting Loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, label='Training Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

# Plotting Accuracy (if accuracy metrics were used)
if train_acc and val_acc:
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_acc, label='Training Accuracy')
    plt.plot(epochs, val_acc, label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
model.evaluate(test_data_generator)

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have the dataset, model, class_names, etc. already defined
correct = 0
incorrect = 0
for i, (image_batch, label_batch) in enumerate(dataset.take(100)):
   # plt.imshow(image_batch[0].numpy().astype('uint8'))
   # plt.axis('off')
   # plt.title(class_names[label_batch[0]])

    prediction = model.predict(image_batch)
    array_str = prediction[0]

    valarray = [float(val) for val in array_str]
    maxval = max(valarray)
    predicted_index = valarray.index(maxval)

    predicted_class = class_names[predicted_index]
    actual_class = class_names[label_batch[0]]

    print("Predicted:", predicted_class)
    print("Actual:", actual_class)

    if predicted_class == actual_class:
      #  print("Correct")
        correct+=1
    else:
       # print("Incorrect")
        incorrect+=1

    #plt.show()
print(f'{correct} and {incorrect}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

# Assuming you have the dataset, model, class_names, etc. already defined

# Lists to hold the true labels and predicted labels
true_labels = []
predicted_labels = []

correct = 0
incorrect = 0

for i, (image_batch, label_batch) in enumerate(dataset.take(2000)):
    prediction = model.predict(image_batch)

    # Assuming model.predict() returns probabilities for each class
    predicted_index = np.argmax(prediction[0])

    predicted_class = class_names[predicted_index]
    actual_class = class_names[label_batch[0]]

 #   print("Predicted:", predicted_class)
 #   print("Actual:", actual_class)

    if predicted_class == actual_class:
        correct += 1
    else:
        incorrect += 1

    # Append the indices of the actual and predicted classes
    true_labels.append(label_batch[0])
    predicted_labels.append(predicted_index)

print(f'{correct} and {incorrect}')

# Convert lists to numpy arrays
true_labels = np.array(true_labels)
predicted_labels = np.array(predicted_labels)

# Generate confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot confusion matrix
plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()
